In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Tutorial 04. Optimization principles

SoftMobility wraps the soft-body rollout in a JAX-friendly objective so that **design parameters** can be optimized by gradient descent.

This tutorial walks through the workflow on a small, fully-determined example: an **asymmetric dumbbell** sinking under gravity.
Two spheres are placed on either side of the origin.
The radius of the left sphere is fixed at `a_0 = 1`; we optimize the radius `a_1` of the right one so that the dumbbell falls with a lateral drift of exactly 1°.

We use:

* `softmobility.SoftBody` for the body,
* `softmobility.FlowBodyRollout` for the trajectory,
* `softmobility.FlowBodyOptimizer` to wrap an Optax optimiser around a scalar objective.


In [ ]:
import jax
import jax.numpy as jnp
import optax

import softmobility as sm
from softmobility.classes import figstyle

figstyle.apply()

## 1. Creating the soft body

The right-sphere radius `a_1` is the only design parameter.
Each sphere carries a gravity-coupled body force `m·g` with `m = (4/3)π a³`:

In [ ]:
yaml_body = """
design_names: [a_1]
input_names: [gravity]
defaults:
  a_1: 0.9
  K: 4.18879  # 4*pi/3

spheres:
  - radius: 1.0  # a_0
    position: [-1, 0, 0]
    force: [K * gravity0, K * gravity1, K * gravity2]
  - radius: a_1  # a_1
    position: [ 1, 0, 0]
    force: [K * a_1**3 * gravity0, K * a_1**3 * gravity1, K * a_1**3 * gravity2]
"""

body = sm.SoftBody(yaml_body, verbose=False)
print(repr(body))

## 2. Time integration

We assume that unit gravity along $-\boldsymbol{e}_3$, and no flow.
The body starts horizontal (along $\boldsymbol{e}_1$).
With unequal radii the heavier sphere experiences more gravitational force but its drag scales differently (mass ∝ r³, drag ∝ r), so the body tilts and drifts laterally.


In [ ]:
dt = 0.005  # time step for simulation
n_steps = 400  # number of time steps
angle_objective = 1 / 180 * jnp.pi  # target angle (radians) for optimization

In [ ]:
rollout = sm.FlowBodyRollout(
    soft_body=body,
    flow=sm.no_flow(),
    input_map={"gravity": sm.gravity_field(g=1)},
)

In [ ]:
positions, _, _ = rollout.rollout(dt=dt, n_steps=n_steps)

final = positions[-1]
horizontal_drift = jnp.sqrt(final[0] ** 2 + final[1] ** 2)
angle = jnp.arctan2(horizontal_drift, -final[2])

print("positions shape  :", positions.shape)
print("final position   :", positions[-1])
print("trajectory angle :", angle, "[radians] (vs objective ", angle_objective, ")")

## 3. Defining the optimization objective 

We want the dumbbell to fall vertically: minimise the **lateral drift** of the body-reference point at the end of the rollout.

The objective signature must be `(rollout, design) -> scalar` so JAX can trace and differentiate it.


In [ ]:
def lateral_drift_squared(rollout, design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    final = positions[-1]
    horizontal_drift = jnp.sqrt(final[0] ** 2 + final[1] ** 2)
    angle = jnp.arctan2(horizontal_drift, -final[2])
    return (angle - angle_objective) ** 2


# Sanity check: gradient at the symmetric point a_1 = 1 should be ~0.
g0 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([0.5]))
g1 = jax.grad(lambda d: lateral_drift_squared(rollout, d))(jnp.array([1.0]))
print(f"∂/∂a_1 of drift² at a_1=0.5 : {float(g0[0]):.4e}")

## 4. Running the optimizer

We use an Adam optimizer, with learning rate 0.01, and 100 steps.
The design parameter `a_1` is clipped to the interval `[0.1, 1.0]`.

In [ ]:
optimizer = sm.FlowBodyOptimizer(
    rollout,
    lateral_drift_squared,
    optax.adam(learning_rate=0.01),
)

optimal_design = optimizer.run(
    init_design=body.design_defaults,
    n_steps=100,
    print_every=10,
    clip_min=0.1,
    clip_max=1.0,
    maximize=False,
)
print(f"\nfinal a_1 = {float(optimal_design[0]):.4f}")

## Step 5 — Compare trajectories

Plot the centre-of-mass trajectory before vs after optimisation.
The initial asymmetric body drifts sideways while the optimised body falls vertically.


In [ ]:
def trajectory(design):
    positions, _, _ = rollout.rollout(
        dt=dt,
        n_steps=n_steps,
        design=design,
    )
    return jnp.asarray(positions)


traj_init = trajectory(body.design_defaults)
traj_opt = trajectory(optimal_design)

fig, ax = figstyle.figure(size="half", aspect=1.0)
ax.plot(
    traj_init[:, 0],
    traj_init[:, 2],
    color=figstyle.COLORS["red"],
    linewidth=2,
    label=f"initial $a_1 = {float(body.design_defaults[0]):.2f}$",
)
ax.plot(
    traj_opt[:, 0],
    traj_opt[:, 2],
    color=figstyle.COLORS["blue"],
    linewidth=2,
    label=f"optimised $a_1 = {float(optimal_design[0]):.3f}$",
)
ax.set_xlabel("x  (lateral drift)")
ax.set_ylabel("z  (vertical fall)")
ax.set_title("Centre-of-mass trajectory before / after optimisation")
ax.legend(loc="upper left", frameon=False)

## Notes

* Here the body is **rigid** (no DOFs); only the design `a_1` changes.
  The same workflow applies when the body has DOFs: the rollout integrates them, the optimizer only sees the scalar objective.
* The optimizer uses `jax.value_and_grad`, so the objective must be built **purely** from JAX operations on the rollout output.